# Génération d’un dataset industriel synthétique contexte GDIZ Bénin

## Objectif

Ce notebook sert de rapport d'ingénierie et de validation pour le dataset synthétique dédié à la maintenance prédictive des machines, inspiré de [AI4I 2020](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) et adapté à un contexte industriel réaliste (type [GDIZ Bénin](https://gdiz-benin.com/fr/)).

Pour ce faire, nous avons modélisé un jumeau numérique comportemental qui intègre :
- les contraintes environnementales du Sud-Bénin,
- les instabilités du réseau électrique local,
- les lois thermo-mécaniques dégradées des machines.

Donc, la base de données est généré pour 30 machines sur une période de 2 ans, avec des enregistrements toutes les 30 minutes.

### 1.1 Contexte climatique et environnemental

Les machines ne fonctionnent pas en vase clos. Au Bénin, elles subissent des changements climatiques cycliques, que nous avons modélisées à travers deux saisons majeures :

1. **La Période de l'Harmattan :** De décembre à février, elle est caractérisée par un vent sec transportant de la poussière désertique très fine.
2. **La Grande Saison des Pluies :** D'avril à juillet, il y a une humidité relative de l'air saturéee, avec des précipitations intenses et fréquentes.

### 1.2 Contexte électrique du réseau SBEE

L'alimentation électrique fournie par la SBEE présente des anomalies critiques pour la survie du parc machine :

* **Chutes de tension et micro-coupures :** Lorsque la tension réseau baisse, les moteurs électriques compensent la perte de puissance en augmentant leur *couple (`torque`)* pour maintenir leur *vitesse de rotation*. Cette hausse de couple engendre un effet Joule *(surchauffe électrique)*.
* **Délestages ou Blackouts :** En cas de coupure brutale, les machines subissent un arrêt d'urgence. Le retour de l'électricité provoque un choc de tension et un à-coup mécanique majeur *(`power_loss_indicator`)*.

### 1.3 Physique de la machine et dépendances temporelles ($n-1$)

Chaque mesure (enregistrée toutes les 5 minutes) dépend directement de l'état au pas précédent ($n-1$). La physique du simulateur repose sur quatre piliers couplés :

1. **Usure :** l'usure *`tool_wear`* ne diminue jamais (sauf lors d'une maintenance). À chaque pas de temps, un incrément de base est appliqué, pondéré par des multiplicateurs environnementaux. Si la machine redémarre suite à un délestage, un choc mécanique est ajouté.
   
2. **Inertie thermique :** la température de la machine (*`process_temperature`*) suit une loi à lissage exponentiel qui augmente en fonction de la friction mécanique et de l'échauffement électrique. En cas d'arrêt, la machine refroidit progressivement pour rejoindre la température ambiante de l'usine.
   
3. **Vibrations :** elles sont modélisées par une composante mécanique à laquelle s'ajoute une dégradation exponentielle liée à l'usure de l'outil. Plus l'outil est usé, plus le jeu mécanique grandit, ce qui augmente la *`vibration`*.
   
4. **Panne :** la probabilité de panne (*`failure`*) est une fonction logistique de l'usure, de la température et des vibrations. Lorsque cette probabilité dépasse un seuil critique, la machine tombe en panne.

### 1.4 Parc de machines

Le dataset simule le comportement d'un modèle générique de machine tournante.
Trois qualités de machines ont été modélisées. De machines de précision légère à des machines lourdes et robustes, chacune avec des sensibilités différentes aux agressions environnementales et électriques.

## Dataset

In [4]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_parquet('../data/raw/gdiz_dataset.parquet')

In [21]:
print(df.head(5))

      machine_id machine_type           timestamp  rotational_speed  \
0  GDIZ_TEX_M_01            L 2024-01-01 00:00:00          0.000000   
1  GDIZ_TEX_M_01            L 2024-01-01 00:30:00        678.581139   
2  GDIZ_TEX_M_01            L 2024-01-01 01:00:00        839.572324   
3  GDIZ_TEX_M_01            L 2024-01-01 01:30:00        750.222499   
4  GDIZ_TEX_M_01            L 2024-01-01 02:00:00        796.840877   

      torque  tool_wear  process_temperature  vibration  target  \
0   0.000000  64.657520            35.313686   0.000000       0   
1  22.307622  64.668213            42.104284   2.262805       0   
2  30.229398  64.680691            48.608053   1.941758       0   
3  21.168159  64.692222            47.525302   2.156718       0   
4  28.082118  64.703596            50.093868   2.250501       0   

   activity_level  benin_season  ambient_temperature  rain_flag   humidity  \
0        0.551389             1            30.313686          0  85.516451   
1        0.470

In [20]:
print(df.groupby('machine_type').apply(lambda x: (x['target'] == 1).sum(), include_groups=False))

machine_type
H    38
L    88
M    44
dtype: int64
